You can download and run this notebook locally, or you can run it for free in a cloud environment using Colab or Sagemaker Studio Lab:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kirbyju/TCIA_Notebooks/blob/main/TCIA_Linux_Data_Retriever_App.ipynb)

[![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github.com/kirbyju/TCIA_Notebooks/blob/main/TCIA_Linux_Data_Retriever_App.ipynb)

# Summary

Access to large, high-quality datasets is essential for researchers to understand disease and precision medicine pathways, especially in cancer. However, HIPAA constraints make sharing medical images outside an individual institution a complex process. [The Cancer Imaging Archive (TCIA)](https://www.cancerimagingarchive.net/) is a public service funded by the National Cancer Institute that addresses this challenge by providing hosting and de-identification services to take major burdens of data sharing off researchers.

**This notebook is focused on basic use cases for identifying TCIA datasets of interest and downloading them using the TCIA Data Retriever application via the command line on a Linux operating system.** If you're interested in additional TCIA notebooks and coding examples, check out the tutorials at https://github.com/kirbyju/TCIA_Notebooks.

# 1 Learn about Available Collections on the TCIA Website

[Browsing Collections](https://www.cancerimagingarchive.net/collections) and viewing [Analysis Results](https://www.cancerimagingarchive.net/tcia-analysis-results/) of datasets on TCIA are the easiest ways to become familiar with what is available. These pages will help you quickly identify datasets of interest, find valuable supporting data that are not available via our APIs (e.g. clinical spreadsheets and non-DICOM radiology data), and answer the most common questions you might have about the datasets.  

# 2 Intro to the TCIA Data Retriever

The main way to download our datasets is to install the TCIA Data Retriever.  This tool provides a number of useful features that you can read more about at https://github.com/TCIA/data-retriever.

**Note:** It's also possible to download these data via Python if you can't or don't want to install the TCIA Data Retriever. This is covered in a [separate notebook](https://github.com/kirbyju/TCIA_Notebooks/blob/main/TCIA_REST_API_Downloads.ipynb).

There are versions of this tool for Windows, Mac and Linux.  Follow these [instructions](https://wiki.cancerimagingarchive.net/display/NBIA/Downloading+TCIA+Images) to install it on your computer.  The application runs with a GUI by default, but you can also run it in command line mode on headless operating systems.  We'll demonstrate how to do this on a Linux Debian distro in the following cells.

In [ ]:
# install required libwebkit2gtk package -- Note: this step should be unnecessary soon
!apt-get update
!apt-get install -y libwebkit2gtk-4.1-0

In [ ]:
# Download and unzip TCIA Data Retriever
!wget -P /content/ https://github.com/TCIA/data-retriever/releases/latest/download/TCIA_Data_Retriever_linux_x86_64.zip
!unzip /content/TCIA_Data_Retriever_linux_x86_64.zip -d /content/

The Data Retriever software works by ingesting a "manifest" file that contains an inventory of the data you'd like to download. Currently we support several types of manifest files.  The goal is to make it so that users don't really need to worry about where the datasets are hosted and can download anything we've published using one application, but for those who are interested in some of the details:

1.   Legacy .tcia files that were generated by the [NBIA software](https://nbia.cancerimagingarchive.net/) that we are currently phasing out of operation for hosting our DICOM datasets.  These are text files that have a few lines of header information and then one row per DICOM Series Instance UID.
2.   Imaging Data Commons (IDC) manifests (.s5cmd), which are replacing NBIA for our open access DICOM data.  These text files are created by the [IDC Portal](https://portal.imaging.datacommons.cancer.gov/explore/), and have some header information followed by S3 URIs that are based on UUIDs generated by IDC.
3. Non-DICOM pathology datasets (e.g. SVS, NDPI, MRXS) files hosted in our [TCIA Pathology Portal](https://www.cancerimagingarchive.net/histopathology-imaging-on-tcia/) use CSV manifests which include an 'imageUrl' column providing each file's download URL.  
4. Controlled access datasets are hosted in other NCI Cancer Research Data Commons (CRDC) repositories such as [General Commons](https://general.datacommons.cancer.gov/#/data).  The manifests for these are CSV files which contain 'drs_uri' or 'File ID' column headers which provide the UUIDs generated by those systems.  

# 3 Downloading a full open access DICOM collection
Let's assume that after [browsing the collections](https://www.cancerimagingarchive.net/collections), you decided you were interested in the [RIDER Breast MRI](https://doi.org/10.7937/K9/TCIA.2015.H1SXNUXL) Collection.  We can find the URL of the manifest to download the full collection by looking at the blue "Download" button on that page.  Then we can download the manifest with the following command.

In [ ]:
!wget https://www.cancerimagingarchive.net/wp-content/uploads/doiJNLP-Fo0H1NtD.tcia

If you look at the file you'll see some configuration information at the top, followed by a list of Series Instance UIDs that are part of the dataset.  These are older manifests based on the NBIA software we've used to manage our DICOM data.  The Data Retriever also works with NCI Imaging Data Commons s5cmd manifests and spreadsheets that contain a column with "SeriesInstanceUID" as a column header.

Let's edit the manifest file to only include the first 3 UIDs in the manifest so that we can demonstrate the download process more quickly.


In [ ]:
with open('doiJNLP-Fo0H1NtD.tcia','r') as firstfile, open('RIDER-Breast-MRI-Sample.tcia','a') as secondfile:
    count = 0
    for line in firstfile:
        # append content to second file
        secondfile.write(line)
        # Stop after header and first 3 series UIDs
        count += 1;
        if count == 9:
            break

Next, let's open the sample manifest file to download the actual DICOM data.  Here are examples for some common use cases:


```
# Public dataset
./TCIA_Data_Retriever --cli -i manifest.tcia -o /data/dicom

# Restricted dataset with credentials
./TCIA_Data_Retriever --cli -i manifest.tcia -o /data/dicom --auth ~/credentials.json

# Large dataset — maximize throughput
./TCIA_Data_Retriever --cli -i manifest.tcia -o /data/dicom -p 20 --max-connections 25

# Slow or unreliable connection
./TCIA_Data_Retriever --cli -i manifest.tcia -o /data/dicom --server-friendly --skip-existing --save-log
```

We'll start with the most basic.

In [ ]:
!/content/TCIA_Data_Retriever-x86_64.AppImage --cli -i /content/RIDER-Breast-MRI-Sample.tcia -o /content/tcia

You should now find that the data have been saved to your machine in a well-organized hierarchy with some useful metadata in the accompanying CSV file which includes licensing information detailing how each scan can be used.


# 4 Downloading using a TCIA Histopathology Portal manifest
For this example, let's assume you went to the [Portal](https://www.cancerimagingarchive.net/histopathology-imaging-on-tcia/) and created a custom cohort using the filters.  You can click the download icon in the top left corner to generate a CSV.  For this example, you could upload that file into Colab by dragging and dropping it in the left side menu.  Alternatively, you can use the small sample manifest in the following cell.

In [ ]:
import csv

output_path = "prostate-mri-histopathology-manifest.csv"

rows = [
    [
        "collection", "collection_doi", "patient_id", "slide_id", "view", "camic_id",
        "wsiimage_url", "has_radiology", "has_genomics", "has_proteomics", "species",
        "cancer_type", "cancer_location", "data_format", "supporting_data_type",
        "modality", "protocol", "par", "magnification", "update"
    ],
    [
        "PROSTATE-MRI", "10.7937/K9/TCIA.2016.6046GUDv", "MIP-PROSTATE-01-0001",
        "MIP-PROSTATE-01-0001", "View", "2060",
        "http://pathdb.cancerimagingarchive.net/index.php/system/files/wsi/ross/Prostate-MRI/converted/Prostate-MRI/MIP-PROSTATE-01-0001.tiff",
        "yes", "no", "no", "Human", "Prostate", "Prostate", "JPG", "N/A",
        "Whole Mount Tissue", "N/A", "N/A", "N/A", "2011-06-30"
    ],
    [
        "PROSTATE-MRI", "10.7937/K9/TCIA.2016.6046GUDv", "MIP-PROSTATE-01-0002",
        "MIP-PROSTATE-01-0002", "View", "2061",
        "http://pathdb.cancerimagingarchive.net/index.php/system/files/wsi/ross/Prostate-MRI/converted/Prostate-MRI/MIP-PROSTATE-01-0002.tiff",
        "yes", "no", "no", "Human", "Prostate", "Prostate", "JPG", "N/A",
        "Whole Mount Tissue", "N/A", "N/A", "N/A", "2011-06-30"
    ],
    [
        "PROSTATE-MRI", "10.7937/K9/TCIA.2016.6046GUDv", "MIP-PROSTATE-01-0003",
        "MIP-PROSTATE-01-0003", "View", "2062",
        "http://pathdb.cancerimagingarchive.net/index.php/system/files/wsi/ross/Prostate-MRI/converted/Prostate-MRI/MIP-PROSTATE-01-0003.tiff",
        "yes", "no", "no", "Human", "Prostate", "Prostate", "JPG", "N/A",
        "Whole Mount Tissue", "N/A", "N/A", "N/A", "2011-06-30"
    ],
]

with open(output_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
    writer.writerows(rows)

print(f"Wrote {len(rows)-1} data rows to {output_path}")

Now let's download the data in our sample manifest.

In [ ]:
!/content/TCIA_Data_Retriever-x86_64.AppImage --cli -i /content/prostate-mri-histopathology-manifest.csv -o /content/tcia

# 5 Downloading controlled access data
In some cases, you must specifically request access to [Collections](https://www.cancerimagingarchive.net/collections/) before you can download them.  Information about how to do this can be found on the homepage for the Collection(s) you're interested in, and on https://www.cancerimagingarchive.net/nih-controlled-data-access-policy/.

Unfortunately, obtaining access to such data is non-trivial and requires the requestor to meet a variety of different criteria to even be eligible.  However, once you gain access to a dataset via dbGaP, you can follow these steps to download it via command line.

Let's say that we're interested in the [RIDER Neuro MRI](http://doi.org/10.7937/K9/TCIA.2015.VOSN3HN1) Collection. As you can see on the Collection page, you must follow the steps outlined in the NIH Controlled Access Data Policy to obtain permission. After you've done this, click the blue Download button on the RIDER Neuro MRI page to save the manifest file to your computer or grab it by using the code shown below.  Then let's edit the manifest file to download only the first three scans.  This time our manifest is a CSV so we want to keep the header row and the first 3 rows of data after that.

In [ ]:
import pandas as pd

# Load the CSV from the URL
url = "https://www.cancerimagingarchive.net/wp-content/uploads/GC_manifest_RIDER-NEURO-MRI_20260326.csv"
df = pd.read_csv(url)

# Keep header + first 3 rows
df_subset = df.head(3)

# Save to a new CSV
df_subset.to_csv("manifest.csv", index=False)

Next, you'll need your **credentials.json** file that you can create at [NCI Data Commons Framework Services](https://nci-crdc.datacommons.io/) with your approved dbGaP account.  

Once signed in to their site, click on the Profile button in the top right corner and then on the “Create an API Key” button.  You'll specify the path to the API key file using the --auth parameter.

In [ ]:
# download the data using the NBIA Data Retriever
# you may need to update the path to your credential file
!/content/TCIA_Data_Retriever-x86_64.AppImage --cli -i '/content/manifest.csv' -o /content/tcia/ --auth '/content/credentials.json'

#!/content/TCIA_Data_Retriever-x86_64.AppImage '/content/manifest.csv' --auth /content/credentials.json

# Acknowledgements
TCIA is funded by the [Cancer Imaging Program (CIP)](https://imaging.cancer.gov/), a part of the United States [National Cancer Institute (NCI)](https://www.cancer.gov/), and is managed by the [Frederick National Laboratory for Cancer Research (FNLCR)](https://frederick.cancer.gov/).

This notebook was created by [Justin Kirby](https://www.linkedin.com/in/justinkirby82/). If you leverage this notebook or any TCIA datasets in your work, please be sure to comply with the [TCIA Data Usage Policy](https://wiki.cancerimagingarchive.net/x/c4hF). In particular, make sure to cite the DOI(s) for the specific TCIA datasets you used in addition to the following paper!

## TCIA Citation

Clark, K., Vendt, B., Smith, K., Freymann, J., Kirby, J., Koppel, P., Moore, S., Phillips, S., Maffitt, D., Pringle, M., Tarbox, L., & Prior, F. (2013). The Cancer Imaging Archive (TCIA): Maintaining and Operating a Public Information Repository. Journal of Digital Imaging, 26(6), 1045–1057. https://doi.org/10.1007/s10278-013-9622-7